# Lily 1.5B Distillation v0.3 — Modal A100 40GB
**Offline SFT distillation of Lily-1.5B from Sarvam-105b-Distill-100k**

Single A100 40GB setup on Modal with inline execution, Unsloth, Flash Attention 2, native BFloat16 precision, step-by-step HF weight checkpointing, and lossless 16-bit model merging.

## Cell 1 — Install Dependencies

In [ ]:
# ==============================================================================
# Cell 1 — Dependency Installation (Modal %uv Fast Package Manager)
# ==============================================================================
# Install latest Unsloth optimized for sequence packing & distillation
%uv pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
%uv pip install wandb -q
%uv pip install flash-attn --no-build-isolation -q
%uv pip install liger-kernel -q

## Cell 2 — Version Check & Hardware Probe

In [ ]:
# ==============================================================================
# Cell 2 — Hardware Probe & Unsloth Verification
# ==============================================================================
# IMPORTANT: 'import unsloth' must be imported first to apply fast CUDA patches
import unsloth
print(f"Unsloth Version : {unsloth.__version__}")

import torch, trl
print(f"TRL Version     : {trl.__version__}")
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Version    : {torch.version.cuda}")

# Probe GPU device properties
p = torch.cuda.get_device_properties(0)
print(f"\nGPU Device     : {p.name}")
print(f"VRAM Capacity  : {p.total_memory / 1e9:.1f} GB")
print(f"Compute        : cc={p.major}.{p.minor}")
print(f"BFloat16       : {'Supported' if p.major >= 8 else 'NOT supported'}")
print(f"FlashAttention2: {'Supported' if p.major >= 8 else 'NOT supported'}")

assert torch.cuda.is_available(), "No GPU detected!"
assert p.major >= 8, f"Requires Ampere+ GPU (A100/H100). Got cc={p.major}.{p.minor}"
print("\n✅ Hardware check passed")

## Cell 3 — Configuration & Distillation Parameters

In [ ]:
# ==============================================================================
# Cell 3 — Configuration & Distillation Hyperparameters
# ==============================================================================
import os, torch

HF_USERNAME = "abhinav0231"

# Model & LoRA Parameters
MODEL_NAME      = "abhinav0231/Lily-1.5b-v0.1"           # Base model (GRPO output)
MAX_SEQ_LENGTH  = 4096                                  # Sequence packing context length
LORA_RANK       = 32                                    # LoRA rank r=32
LORA_ALPHA      = 64                                    # LoRA scaling factor alpha=64

# Teacher Dataset (Sarvam-105b Teacher Trajectories)
DATASET_REPO    = "abhinav0231/Sarvam-105b-Distill-100k"
DATASET_CONFIG  = "chatml"                              # Pre-formatted ChatML split

# Distillation Training Parameters (Optimized for Modal A100 40GB)
NUM_EPOCHS      = 2                                     # 2 full passes over teacher dataset
LEARNING_RATE   = 2e-5                                  # Peak learning rate
WARMUP_STEPS    = 100                                   # 100 linear warmup steps
BATCH_SIZE      = 24                                    # Per-device batch size
GRAD_ACCUM      = 1                                     # Effective batch size = 24 * 1 = 24
SEED            = 42

# Checkpointing & Repos
SAVE_STEPS             = 500
CHECKPOINT_REPO        = f"{HF_USERNAME}/Lily-1.5b-distill-v3-checkpoints"
RESUME_FROM_CHECKPOINT = True

# Output Repositories on Hugging Face Hub
HF_ADAPTER_REPO = f"{HF_USERNAME}/Lily-1.5b-v0.3-adapter"
HF_MERGED_REPO  = f"{HF_USERNAME}/Lily-1.5b-v0.3"

# W&B Config
WANDB_PROJECT   = "superqwen"
WANDB_RUN_NAME  = "lily-1.5b-distill-v3-a100"

# Filesystem Paths on Modal Container
OUTPUT_DIR  = "/root/distill_output"
MERGED_DIR  = "/root/distill_merged"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

DTYPE      = torch.bfloat16
MIXED_PREC = "bf16"

print("Config ready")
print(f"  Model       : {MODEL_NAME}")
print(f"  Dataset     : {DATASET_REPO} ({DATASET_CONFIG})")
print(f"  Epochs      : {NUM_EPOCHS}")
print(f"  Eff batch   : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LoRA r/a    : {LORA_RANK} / {LORA_ALPHA}")
print(f"  Output      : {HF_MERGED_REPO}")
print(f"  Ckpt repo   : {CHECKPOINT_REPO}")

## Cell 4 — Authentication & Auto-Resumption Logic

In [ ]:
# ==============================================================================
# Robust Authentication (Hugging Face)
# ==============================================================================
import os
try:
    from huggingface_hub import login, get_token
except ImportError:
    from huggingface_hub import login, HfFolder
    get_token = HfFolder.get_token

# Retrieve HF Token from all possible sources (os.environ, Colab Secrets, or cached token)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")

# WandB Authentication
try:
    import wandb
    WANDB_TOKEN = os.environ.get("WANDB_API_KEY", "")
    if WANDB_TOKEN and WANDB_TOKEN != "YOUR_WANDB_KEY_HERE":
        wandb.login(key=WANDB_TOKEN, relogin=True)
        os.environ["WANDB_API_KEY"] = WANDB_TOKEN
        print("✅ Authenticated with Weights & Biases")
    else:
        print("ℹ️ WANDB_API_KEY not found. WandB tracking will operate in offline/disabled mode.")
        os.environ["WANDB_DISABLED"] = "true"
except ImportError:
    print("ℹ️ WandB module not installed. Operating without WandB tracking.")
    os.environ["WANDB_DISABLED"] = "true"


## Cell 5 — Load Unsloth Model & Setup LoRA

In [ ]:
# ==============================================================================
# Cell 5 — Unsloth Fast Model Loading & PEFT LoRA Configuration
# ==============================================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = True,
)

tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# Configure LoRA adapters across all 7 linear layers
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    lora_alpha = LORA_ALPHA,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.1f}%")

## Cell 6 — Load Dataset & Configure Sequence Packing

In [ ]:
# ==============================================================================
# Cell 6 — Load Teacher Trajectory Dataset & Sequence Packing Configuration
# ==============================================================================
from datasets import load_dataset

print(f"Loading distillation dataset: {DATASET_REPO} ({DATASET_CONFIG}) ...")
dataset = load_dataset(DATASET_REPO, name=DATASET_CONFIG, split="train")
print(f"✅ Dataset loaded: {len(dataset):,} samples")

# Inspect raw text formatting
sample_text = dataset[0]["text"]
print(f"Sample preview:\n{sample_text[:300]}...\n")

## Cell 7 — Intermediate Weight Checkpoint Push Callback

In [ ]:
# ==============================================================================
# Cell 7 — Custom Trainer Callback for Step Checkpoint Pushing
# ==============================================================================
from transformers import TrainerCallback
from huggingface_hub import upload_folder, HfApi

class CheckpointPushCallback(TrainerCallback):
    """Pushes intermediate checkpoint adapters to HF Hub after every save step."""
    def __init__(self, repo_id: str, token: str):
        self.repo_id = repo_id
        self.token   = token
        if repo_id:
            try:
                HfApi().create_repo(repo_id, token=token, exist_ok=True, private=True)
                print(f"Checkpoint repo ready: {repo_id}")
            except Exception as e:
                print(f"Notice on checkpoint repo create: {e}")

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        ckpt_dir = os.path.join(args.output_dir, f"checkpoint-{step}")
        if not os.path.exists(ckpt_dir):
            return
        print(f"\n>> Step {step}: uploading checkpoint to HF Hub -> {self.repo_id} ...")
        try:
            upload_folder(
                folder_path     = ckpt_dir,
                repo_id         = self.repo_id,
                token           = self.token,
                path_in_repo    = f"checkpoint-{step}",
                commit_message  = f"Distillation checkpoint step {step}",
                ignore_patterns = ["*.lock"],
            )
            print(f">> Step {step} checkpoint uploaded successfully.")
        except Exception as e:
            print(f"WARNING: HF upload failed at step {step}: {e}")

_callbacks = [CheckpointPushCallback(repo_id=CHECKPOINT_REPO, token=HF_TOKEN)]
print("✅ CheckpointPushCallback configured")

## Cell 8 — Configure SFTTrainer with Sequence Packing & Train

In [ ]:
# ==============================================================================
# Cell 8 — SFTTrainer Setup with Sequence Packing (`packing=True`)
# ==============================================================================
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported

wandb.init(
    project = WANDB_PROJECT,
    name    = WANDB_RUN_NAME,
    config  = {
        "model": MODEL_NAME, "lora_rank": LORA_RANK,
        "lr": LEARNING_RATE, "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
        "seq_len": MAX_SEQ_LENGTH, "packing": True,
    }
)

# Configure SFTTrainer with sequence packing enabled
training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    warmup_steps                = WARMUP_STEPS,
    fp16                        = not is_bf16_supported(),
    bf16                        = is_bf16_supported(),
    optim                       = "adamw_torch_fused",
    weight_decay                = 0.0,
    max_grad_norm               = 1.0,
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = "text",
    packing                     = True,      # 2.4x throughput speedup via FlashAttention block-diagonal masking
    logging_steps               = 10,
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 2,
    report_to                   = "wandb",
    run_name                    = WANDB_RUN_NAME,
    seed                        = SEED,
    dataloader_num_workers      = 0,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args          = training_args,
    callbacks     = _callbacks,
)

print(f"\n{'='*60}")
print(f"  Distillation v0.3 Training (A100) | {len(dataset):,} raw samples")
print(f"{'='*60}\n")

stats = trainer.train(resume_from_checkpoint=_resume_ckpt)
print(f"\n✅ Distillation completed in {stats.metrics['train_runtime']/60:.1f} min")
wandb.finish()

## Cell 9 — Save & Push Distillation Adapter

In [ ]:
# ==============================================================================
# Cell 9 — Save & Push Trained Distillation LoRA Adapter
# ==============================================================================
print(f"Saving distillation LoRA adapter locally to {OUTPUT_DIR}/lora_adapter ...")
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

print(f"Pushing adapter to Hugging Face Hub: {HF_ADAPTER_REPO} ...")
model.push_to_hub(HF_ADAPTER_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_ADAPTER_REPO, token=HF_TOKEN)
print(f"✅ Adapter pushed to https://huggingface.co/{HF_ADAPTER_REPO}")

## Cell 10 — Lossless 16-bit Model Merging & Push (`Lily-1.5b-v0.3`)

In [ ]:
# ==============================================================================
# Cell 10 — Lossless 16-bit Model Merging & Hub Deployment (Lily-1.5b-v0.3)
# ==============================================================================
from unsloth import FastLanguageModel

# 1. Reload base model + distillation adapter in full precision BFloat16
print("Loading trained model + adapter in full precision BFloat16 for lossless merge ...")
model_m, tok_m = FastLanguageModel.from_pretrained(
    model_name     = f"{OUTPUT_DIR}/lora_adapter" if os.path.exists(f"{OUTPUT_DIR}/lora_adapter") else HF_ADAPTER_REPO,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,  # Full precision BF16 merge
    load_in_4bit   = False,           # Disable quantization artifacts
)

# 2. Merge LoRA weights into base model
print(f"Merging LoRA weights locally into 16-bit model -> {MERGED_DIR} ...")
model_m.save_pretrained_merged(MERGED_DIR, tok_m, save_method="merged_16bit")
print(f"✅ Merged model saved locally to {MERGED_DIR}")

# 3. Push final merged 16-bit model 'Lily-1.5b-v0.3' to Hugging Face Hub
print(f"Pushing final merged 16-bit model to Hugging Face Hub: {HF_MERGED_REPO} ...")
model_m.push_to_hub_merged(HF_MERGED_REPO, tok_m, save_method="merged_16bit", token=HF_TOKEN)
print(f"\n🎉 Successfully merged and pushed final 16-bit model Lily-1.5b-v0.3 to:")
print(f"   https://huggingface.co/{HF_MERGED_REPO}")